In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import hparam as hp


# load model
from model import EmotionClassifierV1, EmotionClassifierV2, EmotionClassifierHybrid

In [8]:
# load Mini
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(hp.MODEL)
minilm = AutoModel.from_pretrained(hp.MODEL).to(device).eval()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [21]:
# load class
#model = EmotionClassifierV1(hp.NUM_EMOTIONS, hp.input_dim)
#model = EmotionClassifierV2(hp.NUM_EMOTIONS, hp.input_dim, hidden=hp.hdn_dims, dropout=0.25)
model = EmotionClassifierHybrid(hp.NUM_EMOTIONS, hp.input_dim, hidden=hp.hdn_dims, dropout=0.25)
model.load_state_dict(torch.load("emotion_head_best.pt", map_location="cpu"))
model = model.to(device)
model.eval()

EmotionClassifierHybrid(
  (linear): Linear(in_features=384, out_features=28, bias=True)
  (trunk): Sequential(
    (0): Linear(in_features=384, out_features=192, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.25, inplace=False)
    (3): Linear(in_features=192, out_features=192, bias=True)
    (4): GELU(approximate='none')
    (5): Dropout(p=0.25, inplace=False)
  )
  (mlp_head): Linear(in_features=192, out_features=28, bias=True)
)

In [10]:
# Helper functions
def mean_pool(last_hidden, mask):
    m = mask.unsqueeze(-1).float()
    return (last_hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)

@torch.inference_mode()
def get_minilm_hdn(text):
    enc = tokenizer(
        [text],
        padding=True, truncation=True, max_length=hp.MAX_LEN,
        return_tensors='pt',
    ).to(device)

    out = minilm(**enc)
    return mean_pool(out.last_hidden_state, enc['attention_mask'])   # (1, 384)
@torch.inference_mode
def get_class_outputs(input, model):
    return model(input)

In [16]:
texts = ["I am so happy!", "I am so sad!!", "I am sad but also happy", "I am so horny!"]


# I feel bad, but it's for sience!?!!?!?!?
#texts = ["I crave your touch.", "I really want you.", "I desire you.", "I need that so badly."]


In [22]:
for i, text in enumerate(texts):

    print(f"{i} - {text}")

    class_input = get_minilm_hdn(text)
    with torch.inference_mode():
        preds = model(class_input).squeeze(0).cpu()
        preds = torch.sigmoid(preds)


    labels = {
        emotion: round(float(score), 5)
        for emotion, score in zip(hp.EMOTIONS, preds)
    }

    # sorted, filtered view
    for emotion, score in sorted(labels.items(), key=lambda kv: -kv[1]):
        print(f"{emotion:14s} {score:.3f}")

0 - I am so happy!
joy            0.886
excitement     0.364
admiration     0.309
love           0.101
optimism       0.045
gratitude      0.034
pride          0.025
neutral        0.018
approval       0.015
caring         0.006
surprise       0.005
realization    0.005
relief         0.002
desire         0.002
curiosity      0.001
confusion      0.000
amusement      0.000
disappointment 0.000
disapproval    0.000
sadness        0.000
annoyance      0.000
anger          0.000
disgust        0.000
embarrassment  0.000
fear           0.000
grief          0.000
nervousness    0.000
remorse        0.000
1 - I am so sad!!
sadness        0.954
disappointment 0.329
anger          0.070
neutral        0.051
annoyance      0.049
surprise       0.031
nervousness    0.024
fear           0.022
disgust        0.021
optimism       0.019
remorse        0.018
approval       0.017
desire         0.016
grief          0.016
caring         0.012
realization    0.011
embarrassment  0.011
curiosity      0.0

In [23]:
# I will not stop until the model is perfect
# diagnostic_battery.py
CASES = [
    # ============ A. BASELINE ANCHORS (1 per emotion) ============
    # If ANY of these fail, that specific class is broken. Fix the class, not the model.
    # Pass condition: target emotion is TOP-1.
    ('baseline', 'joy',           "I'm so happy!"),
    ('baseline', 'sadness',       "I feel terrible, everything is falling apart."),
    ('baseline', 'anger',         "I am furious right now."),
    ('baseline', 'fear',          "I'm terrified."),
    ('baseline', 'surprise',      "I can't believe that just happened!"),
    ('baseline', 'disgust',       "That's absolutely disgusting."),
    ('baseline', 'love',          "I love you more than anything."),
    ('baseline', 'gratitude',     "Thank you so much, I really appreciate it."),
    ('baseline', 'admiration',    "You're an incredible person, I truly admire you."),
    ('baseline', 'amusement',     "That was hilarious, I can't stop laughing."),
    ('baseline', 'curiosity',     "I really want to know how this works."),
    ('baseline', 'optimism',      "I'm sure things are going to get better."),
    ('baseline', 'pride',         "I'm so proud of what I accomplished today."),
    ('baseline', 'nervousness',   "I'm really nervous about the interview tomorrow."),
    ('baseline', 'disappointment',"I'm so let down by what happened."),
    ('baseline', 'remorse',       "I deeply regret hurting you."),
    ('baseline', 'grief',         "I miss them so much it hurts."),
    ('baseline', 'embarrassment', "I'm so embarrassed, I want to disappear."),
    ('baseline', 'relief',        "Thank god it's finally over."),
    ('baseline', 'realization',   "Oh, I finally get it now."),
    ('baseline', 'confusion',     "I have no idea what's going on."),
    ('baseline', 'disapproval',   "That's just wrong and I don't support it."),
    ('baseline', 'annoyance',     "This is really starting to annoy me."),
    ('baseline', 'approval',      "That's a great plan, I'm all for it."),
    ('baseline', 'caring',        "Please take care of yourself, I'm worried about you."),
    ('baseline', 'desire',        "I really want this more than anything."),
    ('baseline', 'neutral',       "The meeting is at 3 PM in room 4."),
    ('baseline', 'excitement',    "I'm so excited for the trip!"),

    # ============ B. CO-OCCURRENCE (both should appear) ============
    # Pass condition: BOTH expected emotions in TOP-5.
    ('cooccur', 'sadness|joy',            "I'm sad but also happy."),
    ('cooccur', 'nervousness|excitement', "I'm nervous and excited about tomorrow."),
    ('cooccur', 'gratitude|sadness',      "I'm grateful but heartbroken."),
    ('cooccur', 'anger|disappointment',   "I'm angry and disappointed in you."),
    ('cooccur', 'love|fear',              "I love you but I'm scared of losing you."),

    # ============ C. NEGATION / SCOPE ============
    # Pass condition: primary emotion NOT top-1 (for negatives) / still top-1 (for positives).
    ('negation', 'joy',  "I am happy."),
    ('negation', None,   "I am not happy."),                # no pass check
    ('negation', None,   "I'm pretending to be happy."),    # no pass check
    ('negation', None,   "I was happy yesterday but not today."),  # no pass check

    # ============ D. INTENSITY GRADIENTS ============
    # Pass condition: top-1 should stay 'sadness' but magnitude should vary.
    ('intensity', 'sadness', "I'm a little down."),
    ('intensity', 'sadness', "I'm sad."),
    ('intensity', 'sadness', "I'm heartbroken."),

    # ============ E. CONFUSABLE PAIRS ============
    # Tests whether the model distinguishes subtle GoEmotions categories.
    ('confusable', 'annoyance', "I'm mildly annoyed."),
    ('confusable', 'anger',     "I'm furious."),
    ('confusable', 'admiration',"I admire you deeply."),
    ('confusable', 'approval',  "I approve of this decision."),
    ('confusable', 'fear',      "I'm scared."),
    ('confusable', 'nervousness',"I'm nervous."),
    ('confusable', 'desire',    "I want you."),
    ('confusable', 'love',      "I love you."),

    # ============ F. OOD / COLLOQUIAL ============
    # No pass check. These test embedding geometry; failures are known.
    ('ood', None, "I'm horny."),
    ('ood', None, "I'm craving your touch."),
    ('ood', None, "This is lit."),
    ('ood', None, "I'm stoked."),
    ('ood', None, "I'm pissed off."),
    ('ood', None, "That was sick!"),    # ambiguous slang

    # ============ G. NEUTRAL / FACTUAL ============
    # Pass condition: neutral TOP-1, and NO emotion > 0.3 (should abstain).
    ('neutral', 'neutral', "The table is brown."),
    ('neutral', 'neutral', "I went to the store."),
    ('neutral', 'neutral', "Water boils at 100°C."),

    # ============ H. SARCASM / IRONY ============
    # No pass check. Hard for any model — track these as known failure modes.
    ('sarcasm', None, "Great, another Monday."),
    ('sarcasm', None, "Oh wonderful, it's raining again."),
    ('sarcasm', None, "Perfect, just what I needed."),

    # ============ I. PHRASING ROBUSTNESS ============
    # Pass condition: target in TOP-3 (word order shouldn't matter this much).
    ('phrasing', 'joy', "Happy, I am."),
    ('phrasing', 'joy', "It makes me happy."),
    ('phrasing', 'joy', "I feel happiness."),

    # ============ J. CROSS-DOMAIN ============
    # Pass condition: target in TOP-3.
    ('formal',  'joy', "I am exceedingly pleased with this outcome."),
    ('casual',  'excitement', "yo this is amazing"),
    ('poetic',  'joy', "My heart sings with delight."),
]

In [24]:
def run_battery(model, get_embed, scaler=None, EMOTIONS=None, k=5):
    model.eval()
    results = {}   # category -> list of (text, passed, topk)

    for category, target, text in CASES:
        # Get probabilities
        with torch.inference_mode():
            logits = model(get_embed(text))
            if scaler is not None:
                logits = scaler(logits)
            probs = torch.sigmoid(logits.squeeze(0)).cpu()

        # Get top-k
        top = torch.topk(probs, k)
        top_names = [EMOTIONS[i] for i in top.indices.tolist()]
        top_vals  = top.values.tolist()

        # Determine pass/fail
        passed, note = _check(category, target, top_names, top_vals, probs, EMOTIONS)
        results.setdefault(category, []).append((text, passed, top_names, top_vals, note))

    _print_results(results)

def _check(category, target, top_names, top_vals, probs, EMOTIONS):
    if target is None:
        return None, ''

    if category == 'baseline':
        return (top_names[0] == target), ''

    if category == 'cooccur':
        needed = target.split('|')
        hits = [e for e in needed if e in top_names]
        return (len(hits) == len(needed)), f'hits={hits}'

    if category == 'negation':
        if target is None: return None, ''
        return (top_names[0] != target), f'should not be {target}'

    if category == 'intensity':
        return (top_names[0] == target), ''

    if category == 'confusable':
        return (top_names[0] == target), ''

    if category == 'neutral':
        top_val = max(top_vals)
        ok = (top_names[0] == 'neutral') and (top_val < 0.7)
        return ok, f'max_emo={top_vals[1]:.3f} (should be low)'

    if category in ('phrasing', 'formal', 'casual', 'poetic'):
        return (target in top_names[:3]), ''

    return None, ''

def _print_results(results):
    total_pass, total_checked = 0, 0

    print('=' * 80)
    print('DIAGNOSTIC BATTERY')
    print('=' * 80)

    for cat, items in results.items():
        print(f'\n--- {cat.upper()} ---')
        for text, passed, names, vals, note in items:
            mark = {True: '[✓]', False: '[✗]', None: '[?]'}[passed]
            if passed is not None:
                total_checked += 1
                total_pass += int(passed)

            tops = '  '.join(f'{n}:{v:.2f}' for n, v in zip(names[:3], vals[:3]))
            print(f'{mark} "{text}"')
            print(f'      {tops}')
            if note:
                print(f'      note: {note}')

    print()
    print('=' * 80)
    print(f'TOTAL: {total_pass}/{total_checked} passed  '
          f'({100*total_pass/total_checked:.1f}%)')
    print('=' * 80)

run_battery(model, get_minilm_hdn, scaler=None, EMOTIONS=hp.EMOTIONS)

DIAGNOSTIC BATTERY

--- BASELINE ---
[✓] "I'm so happy!"
      joy:0.87  admiration:0.35  excitement:0.34
[✓] "I feel terrible, everything is falling apart."
      sadness:0.85  disappointment:0.36  annoyance:0.10
[✓] "I am furious right now."
      anger:0.60  neutral:0.33  annoyance:0.30
[✓] "I'm terrified."
      fear:1.00  nervousness:0.65  sadness:0.16
[✗] "I can't believe that just happened!"
      neutral:0.55  surprise:0.41  approval:0.10
[✓] "That's absolutely disgusting."
      disgust:0.87  anger:0.34  annoyance:0.34
[✓] "I love you more than anything."
      love:1.00  admiration:0.16  approval:0.03
[✓] "Thank you so much, I really appreciate it."
      gratitude:0.90  admiration:0.32  approval:0.04
[✓] "You're an incredible person, I truly admire you."
      admiration:0.77  gratitude:0.47  love:0.18
[✓] "That was hilarious, I can't stop laughing."
      amusement:0.91  gratitude:0.24  admiration:0.20
[✗] "I really want to know how this works."
      neutral:0.64  curiosit